# Ask CIP v8: bigger search models and bigger readers, tested on our own questions

Ask finds an answer in two steps: **the search** (an embedding model ranks the ~3,000 survey questions by closeness to
the question, with each one's analysts' asks and how long it has run) and **the reader** (a cross-encoder picks one of the
first ~14, or "none of these"). On the Mac, the search has the right survey question in its list for 91-98% of questions
and the reader picks it 84-97% of the time: the reader loses more. This notebook tests both, on our data, with no AI:

1. **Search models**: today's bge-small and 11 others (bge base and large, IBM Granite R2, GTE ModernBERT, Snowflake
   Arctic, mixedbread, e5, Nomic, Qwen3-Embedding-0.6B), each read as its model card says, scored exactly as Ask ranks.
2. **The best two small ones fine-tuned** on our practice questions, with hard negatives (the near misses).
3. **Bigger readers** (IBM Granite reranker R2 and GTE ModernBERT reranker, 149M; BGE reranker base, 278M) trained on
   v7's checked labels, against today's MiniLM L6 (22M) trained the same way.
4. **The winners exported** as Ask loads them (8-bit ONNX, tokenizer, and a `calibration.json` with how to read them and
   thresholds mapped to their scale), timed on one CPU thread.

**Run:** Runtime → Change runtime type → **A100** → Runtime → Run all → pick `v8-search.json`, `v8-tests.json` and
`v7-train.json` from `Downloads/cip-colab`. About 2-2½ hours (~25-30 units); keep the laptop open. `ask-v8.zip` downloads
at the end (send it back: scored on the Mac inside Ask). A step that fails is noted in `results.json` and the rest goes on.
No API key is used anywhere; the "HF_TOKEN" warning is harmless.


In [ ]:
# 1. Libraries, the GPU, the three files (Downloads/cip-colab), and the results kept as we go
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "sentence-transformers", "datasets", "onnx", "onnxruntime", "einops"], check=True)
import gc, json, os, random, shutil, time
import numpy as np, torch
assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → A100, then Run all again"
print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB")
NEEDED = ("v8-search.json", "v8-tests.json", "v7-train.json")
if not all(os.path.exists(f) for f in NEEDED):
    from google.colab import files
    print("Pick v8-search.json, v8-tests.json and v7-train.json from Downloads/cip-colab (hold ⌘)")
    files.upload()
search, tests, train = (json.load(open(f)) for f in NEEDED)
os.makedirs("ask-v8", exist_ok=True)
results = {"embedders": {}, "readers": {}}

def save():
    json.dump(results, open("ask-v8/results.json", "w"), indent=1)

def free():
    gc.collect(); torch.cuda.empty_cache()

print(len(search["docs"]), "survey questions ·", len(search["practice"]), "practice questions ·",
      {t: len(v) for t, v in search["tests"].items()}, "· reader requests", {t: len(tests[t]) for t in ("golden", "fresh", "bank")})


In [ ]:
# 2. The search score, as checked on the Mac with today's model (golden: right one first 57.6% by vectors alone, 98% in
# the first 14 with the asks and standing, as inside Ask): each question's rank of its first right survey question
docs = search["docs"]
at = {d["key"]: i for i, d in enumerate(docs)}
owners = [(i, a) for i, d in enumerate(docs) for a in d["asks"]]
ask_owner = np.array([i for i, _ in owners])
standing = np.array([d["standing"] for d in docs], np.float32)
# settings are chosen on 2,000 practice questions that are not word for word an embedded ask (57% are: they would
# match themselves), never on the tests
said = {a.strip().lower() for d in docs for a in d["asks"]}
tune = [p for p in search["practice"] if p["q"].strip().lower() not in said]
random.Random(8).shuffle(tune)
tune = tune[:2000]
tune_q = {p["q"] for p in tune}
SETS = {"tune": [{"subject": p["subject"], "right": [p["key"]]} for p in tune],
        **{t: [r for r in rows if r["right"]] for t, rows in search["tests"].items()}}
RIGHT = {t: [{at[k] for k in r["right"] if k in at} for r in rows] for t, rows in SETS.items()}

def fused(q, doc_vecs, ask_vecs, w, stand):
    """Closeness as Ask ranks it (api/ask_index.py retrieve): the survey question's own, joined with its nearest ask's
    (weight w), plus its standing (long-running and recent)."""
    near = q @ doc_vecs.T
    if w:
        nearest_ask = near.copy()
        sims = q @ ask_vecs.T
        for row in range(len(q)):
            np.maximum.at(nearest_ask[row], ask_owner, sims[row])
        near = (1 - w) * near + w * nearest_ask
    return near + stand

def first_right(q, doc_vecs, ask_vecs, rights, w, stand, chunk=256):
    """Per question, the rank (1 = first) of its first right survey question, None if not in the top 50."""
    out = []
    for s in range(0, len(q), chunk):
        top = np.argsort(-fused(q[s:s + chunk], doc_vecs, ask_vecs, w, stand), axis=1)[:, :50]
        out += [next((n + 1 for n, d in enumerate(row) if d in right), None) for row, right in zip(top, rights[s:s + chunk])]
    return out

def summary(ranks):
    at_k = lambda k: round(sum(1 for r in ranks if r and r <= k) / len(ranks), 3)
    return {"top1": at_k(1), "top3": at_k(3), "top10": at_k(10), "top14": at_k(14)}

print({t: len(rows) for t, rows in SETS.items()})


In [ ]:
# 3. Search models, each read as its model card says (prefixes; its own pooling), scored as Ask ranks. The asks' weight
# is chosen per model on the tuning questions; the standing is scaled to the model's own spread of closeness (the same
# scale maps its thresholds at export). Today's bge-small goes first: the others are measured against its scale.
from sentence_transformers import SentenceTransformer
BGE_QUERY = "Represent this sentence for searching relevant passages: "
QWEN_TASK = "Instruct: Given an analyst's question about consumer surveys, retrieve the survey question that answers it\nQuery:"
EMBEDDERS = [
    {"name": "BAAI/bge-small-en-v1.5", "tag": "bge-small"},                                  # today's, read as today
    {"name": "BAAI/bge-small-en-v1.5", "tag": "bge-small-instruction", "query": BGE_QUERY},  # the same, with its instruction
    {"name": "BAAI/bge-base-en-v1.5", "tag": "bge-base"},
    {"name": "BAAI/bge-large-en-v1.5", "tag": "bge-large"},
    {"name": "ibm-granite/granite-embedding-small-english-r2", "tag": "granite-small-r2"},
    {"name": "ibm-granite/granite-embedding-english-r2", "tag": "granite-r2"},
    {"name": "Alibaba-NLP/gte-modernbert-base", "tag": "gte-modernbert"},
    {"name": "Snowflake/snowflake-arctic-embed-m-v1.5", "tag": "arctic-m-v1.5", "query": BGE_QUERY},
    {"name": "mixedbread-ai/mxbai-embed-large-v1", "tag": "mxbai-large", "query": BGE_QUERY},
    {"name": "intfloat/e5-base-v2", "tag": "e5-base", "query": "query: ", "doc": "passage: "},
    {"name": "nomic-ai/nomic-embed-text-v1.5", "tag": "nomic-v1.5", "query": "search_query: ", "doc": "search_document: ", "remote": True},
    {"name": "Qwen/Qwen3-Embedding-0.6B", "tag": "qwen3-0.6b", "query": QWEN_TASK},
]

def encode(model, texts, prefix, batch=128):
    return model.encode([prefix + t for t in texts], batch_size=batch, normalize_embeddings=True, convert_to_numpy=True,
                        show_progress_bar=False).astype(np.float32)

def spread(near):   # the typical spread of a question's 50 closest survey questions: the scale thresholds move with
    return float(np.mean(np.sort(near, axis=1)[:, -50:].std(axis=1)))

def pooling_of(model):   # as api/ask_model.py embed names it
    mode = next((m.get_pooling_mode_str() for m in model if hasattr(m, "get_pooling_mode_str")), "")
    return {"cls": "cls", "mean": "mean", "lasttoken": "last"}.get(mode, mode)

def cpu_ms(spec, questions):   # one question at a time on one CPU thread, as the server embeds one (PyTorch, not 8-bit)
    model = SentenceTransformer(spec["name"], device="cpu", trust_remote_code=spec.get("remote", False))
    threads = torch.get_num_threads(); torch.set_num_threads(1)
    started = time.time()
    for q in questions:
        model.encode([spec.get("query", "") + q])
    torch.set_num_threads(threads)
    return round((time.time() - started) / len(questions) * 1000, 1)

BASE, kept = None, {}

def score(model, spec):
    """Every survey question, ask and test question through this model; ranked as Ask ranks; the numbers kept."""
    global BASE
    tag, started = spec["tag"], time.time()
    qp, dp = spec.get("query", ""), spec.get("doc", "")
    doc_vecs = encode(model, [d["embed"] for d in docs], dp)
    ask_vecs = encode(model, [a for _, a in owners], qp)
    q = {t: encode(model, [r["subject"] for r in rows], qp) for t, rows in SETS.items()}
    own_spread = spread(q["tune"] @ doc_vecs.T)
    scale = own_spread / BASE["spread"] if BASE else 1.0
    stand = standing * scale
    tried = {w: summary(first_right(q["tune"], doc_vecs, ask_vecs, RIGHT["tune"], w, stand)) for w in (0.0, 0.3, 0.5, 0.7)}
    w = max(tried, key=lambda w: (tried[w]["top14"], tried[w]["top1"], -abs(w - 0.3)))
    sets = {t: summary(first_right(q[t], doc_vecs, ask_vecs, RIGHT[t], w, stand)) for t in SETS}
    alone = {t: summary(first_right(q[t], doc_vecs, ask_vecs, RIGHT[t], 0.0, 0)) for t in SETS if t != "tune"}
    kept[tag] = {"top1": fused(q["tune"], doc_vecs, ask_vecs, w, stand).max(axis=1),   # for the threshold mapping
                 "nearest_ask": (q["tune"] @ ask_vecs.T).max(axis=1), "doc_vecs": doc_vecs, "ask_vecs": ask_vecs}
    if BASE is None:
        BASE = {"spread": own_spread, "tag": tag}
    results["embedders"][tag] = {
        "name": spec["name"], "params": sum(p.numel() for p in model.parameters()), "dims": int(doc_vecs.shape[1]),
        "pooling": pooling_of(model), "query_prefix": qp, "doc_prefix": dp, "ask_weight": w, "scale": round(scale, 3),
        "weights_tried": tried, "sets": sets, "vectors_alone": alone, "seconds": round(time.time() - started),
        "cpu_ms_question": cpu_ms(spec, [r["subject"] for r in SETS["golden"][:30]])}
    r = results["embedders"][tag]
    print(f"{tag:22} {r['params'] / 1e6:5.0f}M w={w} · golden top1 {sets['golden']['top1']:.0%} top14 {sets['golden']['top14']:.0%} · "
          f"fresh {sets['fresh']['top1']:.0%} / {sets['fresh']['top14']:.0%} · bank {sets['bank']['top1']:.0%} / {sets['bank']['top14']:.0%}"
          f" · tune {sets['tune']['top14']:.0%} · {r['cpu_ms_question']} ms on 1 CPU", flush=True)
    save()

for spec in EMBEDDERS:
    try:
        model = SentenceTransformer(spec["name"], device="cuda", trust_remote_code=spec.get("remote", False),
                                    model_kwargs={"torch_dtype": torch.float16})
        score(model, spec)
    except Exception as exc:   # one model that cannot load or run is noted; the others go on
        results["embedders"][spec["tag"]] = {"name": spec["name"], "error": repr(exc)[:300]}
        print(spec["tag"], "failed:", repr(exc)[:200], flush=True)
        save()
    model = None
    free()


In [ ]:
# 4. The best two small search models (≤ 160M parameters: ~10 ms a question on one CPU) fine-tuned on our practice
# questions: each practice question, and one analyst's ask per survey question, against its survey question with a hard
# negative (a near miss ranked 3rd-12th that is not the answer nor its same wording), one epoch. The tuning questions
# are left out of the training, so the score after is honest.
from datasets import Dataset
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments, losses
from sentence_transformers.training_args import BatchSamplers
SMALL = 160_000_000
specs = {s["tag"]: s for s in EMBEDDERS}
ranked = sorted(((r["sets"]["tune"]["top14"], r["sets"]["tune"]["top1"], tag) for tag, r in results["embedders"].items()
                 if "error" not in r and r["params"] <= SMALL), reverse=True)
for _, _, tag in ranked[:2]:
    spec = specs[tag]
    try:
        model = SentenceTransformer(spec["name"], device="cuda", trust_remote_code=spec.get("remote", False))
        qp, dp = spec.get("query", ""), spec.get("doc", "")
        rows = [(p["subject"], at[p["key"]]) for p in search["practice"] if p["q"] not in tune_q and p["key"] in at]
        rows += [(random.Random(i).choice(d["asks"]), i) for i, d in enumerate(docs) if d["asks"]]
        anchors = encode(model, [t for t, _ in rows], qp)
        doc_vecs = kept[tag]["doc_vecs"]
        rng, anchor, positive, negative = random.Random(9), [], [], []
        for s in range(0, len(rows), 1024):
            top = np.argsort(-(anchors[s:s + 1024] @ doc_vecs.T), axis=1)[:, :12]
            for (text, right), row in zip(rows[s:s + 1024], top):
                same = docs[right]["text"].lower()
                near = [d for d in row[2:] if d != right and docs[d]["text"].lower() != same]
                if near:
                    anchor.append(qp + text); positive.append(dp + docs[right]["embed"]); negative.append(dp + docs[rng.choice(near)]["embed"])
        data = Dataset.from_dict({"anchor": anchor, "positive": positive, "negative": negative})
        args = SentenceTransformerTrainingArguments(output_dir=f"ft-{tag}", num_train_epochs=1, per_device_train_batch_size=64,
                                                    learning_rate=2e-5, warmup_ratio=0.1, bf16=True,
                                                    batch_sampler=BatchSamplers.NO_DUPLICATES, logging_steps=100,
                                                    save_strategy="no", report_to="none")
        SentenceTransformerTrainer(model=model, args=args, train_dataset=data, loss=losses.MultipleNegativesRankingLoss(model)).train()
        model.save(f"ft-{tag}/final")
        tuned = {**spec, "name": f"ft-{tag}/final", "tag": f"{tag}-tuned", "base": spec["name"], "remote": spec.get("remote", False)}
        specs[tuned["tag"]] = tuned
        print(f"{tag}: trained on {len(anchor)} pairs", flush=True)
        score(model.half(), tuned)
        results["embedders"][tuned["tag"]]["base"] = spec["name"]
    except Exception as exc:
        results["embedders"][f"{tag}-tuned"] = {"name": spec["name"], "error": repr(exc)[:300]}
        print(tag, "fine-tune failed:", repr(exc)[:200], flush=True)
    save()
    model = None
    free()


In [ ]:
# 5. Bigger readers, each trained on v7's checked labels as v7 trained L6 (a softmax over "none of these" = 0 and every
# candidate line; the best of its epochs on 900 held-out questions), scored on the golden, fresh and big-exam requests.
# One that beats today's size (L6, retrained here as the yardstick) is shrunk to 8-bit ONNX, read as the server reads it
# (api/ask_model.py read_pick), given its "sure" lead (90% agreement) and timed on one CPU thread.
import onnxruntime, tokenizers
from onnxruntime.quantization import quantize_dynamic, QuantType
from transformers import AutoModelForSequenceClassification, AutoTokenizer
READERS = [("cross-encoder/ms-marco-MiniLM-L6-v2", 3e-5),   # today's size: the yardstick
           ("ibm-granite/granite-embedding-reranker-english-r2", 2e-5),
           ("Alibaba-NLP/gte-reranker-modernbert-base", 2e-5),
           ("BAAI/bge-reranker-base", 2e-5)]

def line_text(line):
    return line.split(". ", 1)[1] if ". " in line[:5] else line

labels = [i["label"] for i in train["items"]]
examples = [(i["q"], [line_text(l) for l in i["lines"]], p) for i, p in zip(train["items"], labels) if p != -1]
random.Random(5).shuffle(examples)
held, train_ex = examples[:900], examples[900:]

def scores(model, tok, question, lines):
    enc = tok([question] * len(lines), lines, truncation=True, max_length=192, padding=True, return_tensors="pt").to("cuda")
    with torch.autocast("cuda", dtype=torch.bfloat16):
        return model(**enc).logits.float().view(-1)

def choose(s):
    best = int(torch.argmax(s))
    return best if float(s[best]) > 0 else None

def agreement(model, tok, rows):
    model.eval()
    with torch.no_grad():
        return float(np.mean([choose(scores(model, tok, q, l)) == p for q, l, p in rows]))

def train_reader(name, lr, epochs=2, per_step=16):
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForSequenceClassification.from_pretrained(name, num_labels=1).to("cuda")
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    steps = epochs * (len(train_ex) // per_step)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / (steps // 10 + 1)) * max(0.0, 1 - s / steps))
    best, best_state = -1.0, None
    for epoch in range(epochs):
        model.train(); random.Random(epoch).shuffle(train_ex); started = time.time()
        for s in range(0, len(train_ex) - per_step + 1, per_step):
            loss = 0
            for q, lines, p in train_ex[s:s + per_step]:
                logits = torch.cat([torch.zeros(1, device="cuda"), scores(model, tok, q, lines)])
                loss = loss + torch.nn.functional.cross_entropy(logits.unsqueeze(0), torch.tensor([0 if p is None else p + 1], device="cuda"))
            (loss / per_step).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step(); opt.zero_grad()
        now = agreement(model, tok, held)
        print(f"  {name} epoch {epoch + 1}: agrees with the labels on {now:.1%} of held-out questions · {time.time() - started:.0f}s", flush=True)
        if now > best:
            best, best_state = now, {k: v.detach().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model.eval(), tok, best

def proxy(test, picks):   # the golden format's phrases (golden, fresh) or the big exam's key
    ok = 0
    for i, p in zip(tests[test], picks):
        if test == "bank":
            ok += p is not None and i["keys"][p] == i["answer_key"]
        else:
            ok += (p is None) if i["want"] == "none" else (p is not None and any(w in i["texts"][p].lower() for w in i["want"]))
    return ok

def torch_picks(model, tok, test):
    model.eval()
    with torch.no_grad():
        return [choose(scores(model, tok, i["q"], [line_text(l) for l in i["lines"]])) for i in tests[test]]

def shrink(model, tok, folder):
    """The reader as 8-bit ONNX in folder, and two readers of it as api/ask_model.py read_pick reads (pick, lead): one on
    every CPU thread (to score), one on one thread (to time)."""
    os.makedirs(folder, exist_ok=True)
    dummy = tok(["a question"], ["a survey question"], return_tensors="pt")
    names = [n for n in ("input_ids", "attention_mask", "token_type_ids") if n in dummy]   # ModernBERT: no type ids
    cpu = model.float().cpu().eval()
    for attempt in ("as loaded", "eager"):
        try:
            if attempt == "eager":
                cpu.config._attn_implementation = "eager"
            torch.onnx.export(cpu, tuple(dummy[n] for n in names), "full.onnx", input_names=names, output_names=["logits"],
                              dynamic_axes={n: {0: "batch", 1: "tokens"} for n in names} | {"logits": {0: "batch"}},
                              opset_version=17, dynamo=False)
            break
        except Exception as exc:
            print("  export", attempt, "failed:", repr(exc)[:150])
            if attempt == "eager":
                raise
    quantize_dynamic("full.onnx", f"{folder}/model_optimized.onnx", weight_type=QuantType.QInt8)
    tok.save_pretrained("tok_tmp"); shutil.copy("tok_tmp/tokenizer.json", f"{folder}/tokenizer.json")
    words = tokenizers.Tokenizer.from_file(f"{folder}/tokenizer.json")
    words.enable_truncation(192); words.enable_padding(pad_id=tok.pad_token_id or 0, pad_token=tok.pad_token or "[PAD]")

    def reader(threads):
        options = onnxruntime.SessionOptions()
        if threads:
            options.intra_op_num_threads = options.inter_op_num_threads = threads
        session = onnxruntime.InferenceSession(f"{folder}/model_optimized.onnx", options, providers=["CPUExecutionProvider"])
        inputs = {i.name for i in session.get_inputs()}

        def read(question, lines):
            coded = words.encode_batch([(question, t) for t in lines])
            feed = {"input_ids": np.array([c.ids for c in coded], np.int64), "attention_mask": np.array([c.attention_mask for c in coded], np.int64),
                    "token_type_ids": np.array([c.type_ids for c in coded], np.int64)}
            choices = np.concatenate([[0.0], session.run(None, {k: v for k, v in feed.items() if k in inputs})[0].reshape(-1)])
            best, second = np.argsort(-choices)[:2]
            return (None if best == 0 else int(best) - 1), float(choices[best] - choices[second])
        return read
    model.to("cuda")
    return reader(0), reader(1)

yardstick = None
for name, lr in READERS:
    short = name.split("/")[-1]
    try:
        model, tok, agree = train_reader(name, lr)
        r = {"name": name, "params": sum(p.numel() for p in model.parameters()), "agrees": agree,
             **{t: proxy(t, torch_picks(model, tok, t)) for t in ("golden", "fresh", "bank")},
             "of": {t: len(tests[t]) for t in ("golden", "fresh", "bank")}}
        yardstick = yardstick or r
        if r is not yardstick and agree > yardstick["agrees"] + 0.005:   # better than today's size: worth trying in Ask
            fast, timed = shrink(model, tok, f"ask-v8/readers/{short}")
            reads = [fast(q, l) for q, l, p in held[:450]]
            right = [x[0] == p for x, (q, l, p) in zip(reads, held)]
            sure = next((m for m in (0.5, 1, 1.5, 2, 2.5, 3, 4, 5) if np.mean([a for a, x in zip(right, reads) if x[1] >= m] or [0]) >= 0.9), 5)
            started = time.time()
            for i in tests["golden"][:40]:
                timed(i["q"], [line_text(l) for l in i["lines"]])
            r.update({"8-bit agrees": float(np.mean(right)), "sure": sure, "sure_share": float(np.mean([x[1] >= sure for x in reads])),
                      "ms_question_1cpu": round((time.time() - started) / 40 * 1000)})
            json.dump({"sure": sure, "length": 192, "pad_token": tok.pad_token or "[PAD]",
                       "reads": "api/ask_model.py describe_candidate, the candidate as the AI reads it",
                       "made": f"scripts/colab/train_search_model_v8.ipynb: {name} trained on v7's checked labels, 8-bit"},
                      open(f"ask-v8/readers/{short}/calibration.json", "w"))
        results["readers"][short] = r
        print(f"{short}: agrees {agree:.1%} · golden {r['golden']}/{r['of']['golden']} · fresh {r['fresh']}/{r['of']['fresh']} · "
              f"bank {r['bank']}/{r['of']['bank']}" + (f" · 8-bit {r['8-bit agrees']:.1%}, sure {r['sure']}, {r['ms_question_1cpu']} ms on 1 CPU" if "sure" in r else ""), flush=True)
    except Exception as exc:
        results["readers"][short] = {"name": name, "error": repr(exc)[:300]}
        print(short, "failed:", repr(exc)[:200], flush=True)
    save()
    model = None
    free()


In [ ]:
# 6. The search models worth trying in Ask (the best small one, ≤ 160M, and the best up to 400M, if not today's), as
# api/ask_model.py loads one: 8-bit ONNX of the transformer (its last hidden states: embed pools them), the tokenizer, and
# calibration.json with how to read it and today's thresholds mapped to its scale: an absolute one (nothing close, a
# verified match) by the same share of tuning questions on each side, an added one (standing, area, gate) by its spread.
# Checked: the 8-bit file, read as the server reads it, gives PyTorch's vectors; timed on one CPU thread.
import onnxruntime, tokenizers
from onnxruntime.quantization import quantize_dynamic, QuantType
today = search["bge_small"]   # bge-small's thresholds, as Ask uses them

class Hidden(torch.nn.Module):   # the transformer alone: its last hidden states
    def __init__(self, inner):
        super().__init__()
        self.inner = inner

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        extra = {} if token_type_ids is None else {"token_type_ids": token_type_ids}
        if hasattr(self.inner.config, "use_cache"):
            extra["use_cache"] = False
        return self.inner(input_ids=input_ids, attention_mask=attention_mask, **extra).last_hidden_state

def server_embed(folder, cal, texts, threads=0):   # api/ask_model.py embed, line for line
    words = tokenizers.Tokenizer.from_file(f"{folder}/tokenizer.json")
    words.enable_truncation(cal["length"]); words.enable_padding(pad_id=words.token_to_id(cal["pad_token"]) or 0, pad_token=cal["pad_token"])
    options = onnxruntime.SessionOptions()
    if threads:
        options.intra_op_num_threads = options.inter_op_num_threads = threads
    session = onnxruntime.InferenceSession(f"{folder}/model_optimized.onnx", options, providers=["CPUExecutionProvider"])
    inputs = {i.name for i in session.get_inputs()}
    out = []
    for t in texts:
        c = words.encode(t)
        ids, mask = np.array([c.ids], np.int64), np.array([c.attention_mask], np.int64)
        hidden = session.run(None, {k: v for k, v in {"input_ids": ids, "attention_mask": mask, "token_type_ids": np.zeros_like(ids)}.items() if k in inputs})[0]
        v = hidden[:, 0] if cal["pooling"] == "cls" else hidden[np.arange(1), mask.sum(1) - 1] if cal["pooling"] == "last" else (hidden * mask[..., None]).sum(1) / mask.sum(1, keepdims=True)
        out.append(v[0] / np.linalg.norm(v[0]))
    return np.array(out, np.float32)

def mapped(value, base, own):   # the same share of tuning questions below the threshold, on the model's own scale
    return round(float(np.quantile(own, float(np.mean(base < value)))), 4)

ok = {t: r for t, r in results["embedders"].items() if "error" not in r}
rank = lambda t: (ok[t]["sets"]["tune"]["top14"], ok[t]["sets"]["tune"]["top1"])
picked = [max((t for t in ok if ok[t]["params"] <= limit), key=rank, default=None) for limit in (SMALL, 400_000_000)]
for tag in dict.fromkeys(t for t in picked if t and t != BASE["tag"]):
    spec, r = specs[tag], ok[tag]
    try:
        if r["pooling"] not in ("cls", "mean", "last"):
            raise ValueError(f"pooling {r['pooling']} is not one the server reads")
        model = SentenceTransformer(spec["name"], device="cpu", trust_remote_code=spec.get("remote", False))
        folder = f"ask-v8/embedders/{tag}"
        os.makedirs(folder, exist_ok=True)
        tok = model.tokenizer
        dummy = tok(["What share of shoppers bought from Amazon?"], return_tensors="pt")
        names = [n for n in ("input_ids", "attention_mask", "token_type_ids") if n in dummy]
        torch.onnx.export(Hidden(model[0].auto_model).eval(), tuple(dummy[n] for n in names), "embed.onnx", input_names=names,
                          output_names=["last_hidden_state"], opset_version=17, dynamo=False,
                          dynamic_axes={n: {0: "batch", 1: "tokens"} for n in names} | {"last_hidden_state": {0: "batch", 1: "tokens"}})
        quantize_dynamic("embed.onnx", f"{folder}/model_optimized.onnx", weight_type=QuantType.QInt8)
        tok.save_pretrained("tok_tmp"); shutil.copy("tok_tmp/tokenizer.json", f"{folder}/tokenizer.json")
        base, own = kept[BASE["tag"]], kept[tag]
        cal = {"pooling": r["pooling"], "query_prefix": r["query_prefix"], "doc_prefix": r["doc_prefix"], "length": 512,
               "pad_token": tok.pad_token or "[PAD]", "ask_weight": r["ask_weight"],
               "near_floor": mapped(today["near_floor"], base["top1"], own["top1"]),
               "verified_near": mapped(today["verified_near"], base["nearest_ask"], own["nearest_ask"]),
               "gate": round(today["gate"] * r["scale"], 4), "standing": round(today["standing"] * r["scale"], 4),
               "area_boost": round(today["area_boost"] * r["scale"], 4),
               "made": f"scripts/colab/train_search_model_v8.ipynb: {r.get('base', spec['name'])}{', fine-tuned on our practice questions' if 'base' in r else ''}, 8-bit"}
        json.dump(cal, open(f"{folder}/calibration.json", "w"), indent=1)
        sample = [r_["subject"] for r_ in SETS["tune"][:200]]
        torch_vecs = encode(model, sample, cal["query_prefix"])
        onnx_vecs = server_embed(folder, cal, [cal["query_prefix"] + s for s in sample])
        started = time.time()
        server_embed(folder, cal, [cal["query_prefix"] + s for s in sample[:40]], threads=1)
        r["export"] = {"folder": folder, "calibration": cal, "same_as_pytorch": round(float(np.mean(np.sum(torch_vecs * onnx_vecs, axis=1))), 4),
                       "ms_question_1cpu_8bit": round((time.time() - started) / 40 * 1000, 1),
                       "mb": round(os.path.getsize(f"{folder}/model_optimized.onnx") / 1e6)}
        print(tag, json.dumps(r["export"], indent=1), flush=True)
    except Exception as exc:
        r["export"] = {"error": repr(exc)[:300]}
        print(tag, "export failed:", repr(exc)[:200], flush=True)
    save()
    model = None
    free()


In [ ]:
# 7. The zip (results.json; each exported reader and search model a folder Ask loads with CIP_ASK_READER_DIR or
# CIP_ASK_EMBED_DIR), then the numbers in short
import zipfile
save()
with zipfile.ZipFile("ask-v8.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, names in os.walk("ask-v8"):
        for n in names:
            z.write(os.path.join(root, n))
print(f"ask-v8.zip: {os.path.getsize('ask-v8.zip') / 1e6:.0f} MB\n")
print("Search models (right one first / in the first 14):")
for tag, r in results["embedders"].items():
    if "error" in r:
        print(f"  {tag:24} failed: {r['error'][:80]}")
    else:
        s = r["sets"]
        print(f"  {tag:24} {r['params'] / 1e6:5.0f}M · golden {s['golden']['top1']:.0%}/{s['golden']['top14']:.0%} · fresh {s['fresh']['top1']:.0%}/"
              f"{s['fresh']['top14']:.0%} · bank {s['bank']['top1']:.0%}/{s['bank']['top14']:.0%} · {r['cpu_ms_question']} ms (PyTorch, 1 CPU)")
print("Readers (right picks on the requests that reach the AI step):")
for short, r in results["readers"].items():
    print(f"  {short:44} " + (f"failed: {r['error'][:80]}" if "error" in r else
          f"agrees {r['agrees']:.1%} · golden {r['golden']}/{r['of']['golden']} · fresh {r['fresh']}/{r['of']['fresh']} · bank {r['bank']}/{r['of']['bank']}"
          + (f" · exported, {r['ms_question_1cpu']} ms on 1 CPU" if "sure" in r else "")))
from google.colab import files
files.download("ask-v8.zip")
print("Done: send ask-v8.zip. Runtime → Disconnect and delete runtime stops the units.")
